<a href="https://colab.research.google.com/github/lifan149/notes/blob/main/fastai/Practical-Deep-Learning-for-Coders/dlfc_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

根据[fastbook第四节](https://github.com/fastai/fastbook/blob/master/04_mnist_basics.ipynb)记录的笔记

In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastai.vision.all import *
from fastbook import *

matplotlib.rc('image', cmap='Greys')

根据教程尝试创建一个可以将任何图像分类为 3 或 7 的模型。下载一个仅包含这些数字图像的 MNIST 示例

In [ ]:
path = untar_data(URLs.MNIST_SAMPLE)

In [ ]:
#hide
Path.BASE_PATH = path

通过 fastai 添加的方法 ls 来查看这个目录中的内容。此方法返回一个名为 L 的特殊 fastai 类的对象，该类具有与 Python 内置列表相同的功能，以及更多功能。它的一个方便的功能是，在打印时，它会在列出项目本身之前显示项目计数（如果超过 10 个项目，它只显示前几个项目）：

In [ ]:
path.ls()

MNIST 数据集遵循机器学习数据集的常见布局：训练集和验证集（和/或测试集）的单独文件夹。

In [ ]:
# 查看训练集的内容
(path/'train').ls()

有一个 3 的文件夹和一个 7 的文件夹。在机器学习术语中，我们说“3”和“7”是此数据集中的标签 （或目标）

In [ ]:
threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()
threes

一张手写数字 3 的图片，取自手写数字 MNIST 数据集

In [ ]:
im3_path = threes[1]
im3 = Image.open(im3_path)
im3

在计算机中，一切都表示为数字。要查看构成此图像的数字，我们必须将其转换为 NumPy 数组或 PyTorch 张量 。

**本质上，NumPy 数组（ndarray）和 PyTorch 张量（Tensor）在核心结构上都是多维数组**

**PyTorch 张量在 NumPy 数组的基础上，额外提供了 GPU 加速和自动求导的功能，使其更适合深度学习任务。**

将图像的一部分转为NumPy 数组

4:10 (对于行): 选取从索引为 4 的行开始，直到但不包括索引为 10 的行。换句话说，它会选择索引为 4, 5, 6, 7, 8, 9 的这 6 行。

4:10 (对于列): 这表示选取从索引为 4 的列开始，直到但不包括索引为 10 的列。同样地，它会选择索引为 4, 5, 6, 7, 8, 9 的这 6 列。

NumPy 从上到下、从左到右编制索引，因此此部分位于图像的左上角

In [ ]:
array(im3)[4:10,4:10]

 PyTorch 张量也是如此

In [ ]:
tensor(im3)[4:10,4:10]

## 关于图像数据的多维数组
---

### **1. 彩色图像的三维数组结构**
- **维度定义**  
  彩色图像通常表示为 **三维数组**，结构为 `(高度, 宽度, 颜色通道)`，例如 RGB 图像的通道数为 3（红、绿、蓝）。  
  - **示例**：一个分辨率为 `1920x1080` 的 RGB 图像，其数组形状为 `(1080, 1920, 3)`。  
  - **通道顺序**：OpenCV 中默认通道顺序为 **BGR**（蓝、绿、红），而非 RGB。

- **像素值含义**  
  每个通道的像素值范围通常为 **0-255**，表示颜色强度：  
  - `0`：表示该通道无贡献（如红色通道为 0 时，无红色成分）。  
  - `255`：表示该通道最大强度。

---

### **2. 灰度图像的二维数组结构**
- **维度定义**  
  灰度图像仅包含亮度信息，因此简化为 **二维数组**，结构为 `(高度, 宽度)`。  
  - **示例**：同样分辨率 `1920x1080` 的灰度图像，数组形状为 `(1080, 1920)`。

- **像素值含义**  
  每个像素值直接表示亮度，范围仍为 **0-255**：  
  - `0`：纯黑色。  
  - `255`：纯白色，中间值表示不同灰度等级。

---

### **3. 内存存储与数组组织**
- **多维数组的线性化**  
  尽管逻辑上是多维结构，但计算机内存是线性地址空间。多维数组通过 **行优先顺序** 存储，例如：  
  - 二维数组 `B[a][b]` 的内存布局为连续的 `a*b` 个元素。  
  - 三维数组（如 RGB 图像）则按通道顺序依次存储每个像素的红、绿、蓝值。

---

### **4. 应用场景对比**
- **彩色图像**  
  适用于需要颜色信息的场景（如物体识别、视频处理），但数据量较大（3 倍于灰度图像）。  
- **灰度图像**  
  常用于简化计算（如边缘检测、二值化处理），或颜色无关的分析任务。

---

### **示例代码（Python + OpenCV）**
```python
import cv2

# 读取彩色图像（三维数组）
color_img = cv2.imread("image.jpg")  # 形状为 (H, W, 3)，通道顺序为 BGR
print(color_img.shape)  # 输出：(高度, 宽度, 3)

# 转换为灰度图像（二维数组）
gray_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY)  # 形状变为 (H, W)
print(gray_img.shape)  # 输出：(高度, 宽度)
```

---

**总结**：  
- 彩色图像通过三维数组表示颜色通道，灰度图像通过二维数组表示亮度。  
- 像素值范围 0-255 是数字图像的通用标准，与硬件（如显示器、传感器）的 8 位分辨率相关。


> 如果图像只包含 **0 和 255** 的像素值，则该图像属于 **二值图像**（Binary Image）。其特点是每个像素仅用两种值表示：  
- **0**：通常表示黑色（背景）。  
- **255**：通常表示白色（前景或目标物体）。  
>
> ---
>
> ### **关键特性与用途**
1. **图像分割**：通过阈值处理将灰度图像转换为二值图像，用于区分目标与背景（如边缘检测）。  
2. **简化计算**：仅保留关键信息，减少数据量，常用于OCR、条形码识别等场景。  
3. **形态学操作**：如腐蚀、膨胀等处理通常基于二值图像实现。  
>
> ---
>
> ### **与其他图像类型的对比**
- **灰度图像**：像素值范围为 **0-255**（连续灰度），而二值图像是灰度图像的特例。  
- **彩色图像**：需要三个通道（如RGB）存储颜色信息，而二值图像为单通道。  


对数组进行切片，只选择数字顶部的部分，然后使用 Pandas DataFrame 通过渐变对值进行颜色编码，这清楚地向我们展示了图像是如何从像素值创建的：

In [ ]:
#hide_output
im3_t = tensor(im3)
df = pd.DataFrame(im3_t[4:15,4:22])
df.style.set_properties(**{'font-size':'6pt'}).background_gradient('Greys')

可以看到，背景白色像素存储为数字 0，黑色像素存储为数字 255，灰色阴影介于两者之间。整个图像包含 28 个横向像素和 28 个向下像素，总共 784 个像素

**计算机如何识别3和7呢**

尝试使用像素相似度。找到 3/7 的每个像素的平均像素值，将得到两组平均值，定义我们可以称之为“理想”的 3 和 7。然后，要将图像分类为一个数字或另一个数字，我们可以看到图像与这两个理想数字中的哪一个最相似，这是一个很好的基线。

> 在机器学习和数据科学领域，**"baseline" (基线)** 指的是一个**简单、容易实现但性能可能不是最优的模型或方法**，用于作为比较的基准。它的作用是：
>>
* **提供一个起步点：** 在尝试更复杂、更精密的模型之前，先建立一个简单的基线模型，了解最基本的方法能够达到的性能水平。
* **衡量改进：** 后续开发的更复杂的模型或技术，其性能应该显著优于这个基线模型，才能证明其有效性。如果一个复杂的模型甚至不如简单的基线模型，那说明这个复杂模型可能存在问题。
* **理解问题的难度：** 基线模型的性能可以帮助我们初步了解分类任务的难易程度。如果一个非常简单的基线模型就能达到很高的准确率，那可能说明这个问题本身就比较容易。
>
> 总而言之，这里的 "baseline" 指的是一个简单但合理的起始方法，用于建立一个初步的分类性能水平，并作为未来更复杂方法进行比较的参考。

使用 Python 列表推导式来创建图像张量的普通列表

> 注意：列表推导式：列表和字典推导式是 Python 的一个很棒的功能。许多 Python 程序员每天都在使用它们，包括本书的作者 — 它们是“惯用 Python”的一部分。但是来自其他语言的程序员可能以前从未见过它们。只需在网上搜索一下，就有很多很棒的教程，所以我们现在不会花很长时间讨论它们。下面是一个快速说明和示例，可帮助您入门。列表推导式如下所示： new_list = [f(o) for o in a_list if o>0] .这将返回 a_list 中大于 0 的所有元素，然后将其传递给函数 f。这里有三个部分：你要迭代的集合 （a_list）、一个可选的过滤器 （if o>0） 和对每个元素执行的作 （f（o））。它不仅编写时间更短，而且比使用循环创建相同列表的替代方法要快得多。

In [ ]:
seven_tensors = [tensor(Image.open(o)) for o in sevens]
three_tensors = [tensor(Image.open(o)) for o in threes]
len(three_tensors),len(seven_tensors)

seven_tensors 和three_tensors 实际上是列表，图像张量的列表，打印出的结构虽然看上去和三维数组打印的结构一样，但是它们不是三维数组

使用 fastai 的 show_image 函数来显示它

In [ ]:
show_image(three_tensors[1]);

对于每个像素位置，我们想要计算该像素强度的所有图像的平均值。为此，我们首先将此列表中的所有图像组合成一个三维张量。描述此类张量的最常见方式是将其称为 rank-3 张量。

可以将二维张量相信成一张纸，所谓合成三维张量就像把多张形状相同的纸（二维张量）一层一层地堆叠起来形成一个长方体。

PyTorch 中的某些作（例如取平均值）需要我们将整数类型转换为浮点类型。由于我们稍后会需要它，因此我们现在还将 stacked tensor 转换为 float。在 PyTorch 中进行强制转换非常简单，只需键入要强制转换的类型的名称，并将其视为一种方法即可。

通常，当图像为浮点数时，像素值预期在 0 和 1 之间，因此我们在这里也要除以 255：

In [ ]:
stacked_sevens = torch.stack(seven_tensors).float()/255
stacked_threes = torch.stack(three_tensors).float()/255
stacked_threes.shape

根据上面执行结果 (6131, 28, 28)

它的形状是 (6131, 28, 28)。这个形状告诉我们：
* 第一个轴（维度）的长度是 6131。这通常表示我们有 6131 个样本，在这个上下文中是 6131 张图像。
* 第二个轴的长度是 28。这通常表示每张图像的高度是 28 像素。
* 第三个轴的长度是 28。这通常表示每张图像的宽度是 28 像素。

对于 PyTorch 来说，一个形状为 (6131, 28, 28) 的张量仅仅是在内存中存储的一串数字，按照这个特定的多维结构排列。**张量的形状只是数据的组织结构，而每个维度代表什么含义，是由我们根据实际应用场景来解释的**


张量形状的长度

In [ ]:
len(stacked_threes.shape)

张量的**秩就是张量的维度数量，也就是 `shape` 元组的长度。**

* 对于一个标量（例如 `5`），其 `shape` 是 `()`（空元组），因此秩是 0。
* 对于一个向量（例如 `[1, 2, 3]`），其 `shape` 是 `(3,)`，因此秩是 1。
* 对于一个矩阵（例如 `[[1, 2], [3, 4]]`），其 `shape` 是 `(2, 2)`，因此秩是 2。
* 对于我们例子中的图像张量，其 `shape` 是 `(6131, 28, 28)`，因此秩是 3（它是一个 rank-3 张量或三维张量）。

**理解张量的两个基本属性：**
1.  **形状 (shape):** 描述了张量在每个维度上的大小，是理解数据组织方式的关键。
2.  **秩 (rank):** 描述了张量的维度数量。


**rank 是张量中的轴或维度的数量;shape 是张量每个轴的大小**

---

**注意："维度"（dimension）在不同语境下的不同含义**

1. **物理空间中的“维度” vs. 张量的“秩”（rank）**
* 物理空间的三维性：当我们说“三维空间”时，是指描述一个点的位置需要 三个独立参数（例如 x, y, z 坐标）。这里的“维度”指的是 空间方向的数量，每个方向对应一个轴（axis），且每个轴的长度（元素数量）可能不同。

* 张量的秩（rank）：在 PyTorch 中，`v.ndim` 返回张量的 秩（即轴的个数），而非每个轴的长度。例如：

* 一个三维空间位置向量 `v = [1, 2, 3]` 在 PyTorch 中是 一维张量，因为它的 `ndim=1`，仅有一个轴，该轴的长度为 3（`shape=[3]`）。

* 矩阵（二维张量）的 `ndim=2`，因为它有两个轴（行和列）。


> 关键区分：  
> - 物理中的“三维”对应张量的 某个轴的长度（如 `shape=[3]`）；  
> - 张量的“维度”（`ndim`）实际是 秩（轴的个数）。

2. **术语歧义的根源**

* “维度”一词的多义性：

  * 轴的数量（Rank）：在张量中，`ndim` 表示秩（如矩阵是二维张量）。

  * 轴的长度（Size/Shape）：在物理中，“三维”指每个轴的长度（如 `shape=[3]`）。

* 混淆示例：

  * 若有人说“三维张量”，可能指：

    1. 秩为 3 的张量（如 `shape=[2,3,4]`，三个轴）；
    2. 物理空间中某个轴的长度为 3 的张量（如 `shape=[3]`，秩为 1）。

3. **如何避免混淆？**
建议使用无歧义的术语：
* 秩（Rank）：张量的轴数（`ndim`）。

* 轴（Axis）：张量的某个独立方向（如矩阵的行或列）。

* 长度（Shape）：每个轴上的元素数量（如 `shape=[3,4]` 表示第一轴长 3，第二轴长 4）。


举例说明：
* 物理位置向量 `v = [x, y, z]` 在 PyTorch 中：

* 秩为 1（一维张量）；

* 轴长度为 3（`shape=[3]`）；

* 它的“三维性”体现在轴的长度上，而非张量的秩。


4. **总结：理解张量的核心属性**
* `ndim`（秩）：轴的个数（如向量=1，矩阵=2）。

* `shape`（形状）：每个轴的长度（如 `shape=[3,4]` 表示二维张量，第一轴长 3，第二轴长 4）。

* 物理意义的“维度”：通常对应 `shape` 中的某个值，而非 `ndim`。


> 实际应用：在 PyTorch 中，讨论张量时应明确使用 秩、轴、形状，而非物理中的“维度”概念，以避免误解。

也可以使用 ndim 获取张量的秩：

In [ ]:
stacked_threes.ndim

对于每个像素位置，这将计算该像素在所有图像上的平均值。结果将是每个像素位置一个值，或单个图像

In [ ]:
mean3 = stacked_threes.mean(0)
show_image(mean3);